
# Tokenizer Script-Routing Pilot — 문자권 분기 로직 병행 검증

**이 노트북이 하는 일과 하지 않는 일을 먼저 밝힙니다.**

업로드해주신 `팬덤 텍스트 분석 결과 보고서(v7) — 별도 보고서: 토크나이저 사용
현황 정리 + 언어(문자권)별 워드클라우드`(`docs/TOKENIZER_WORDCLOUD_REPORT.md`로
정리됨)는 **이미 실제 코드를 라이브 코퍼스 10,020건 전체에 실행해서 얻은
결과**입니다. 그 보고서가 언급하는 `run_lda_v6.py`,
`run_lda_v6_live_reference_v7.py`의 실제 소스, 정확한 불용어 목록
(STOPWORDS 41종 등), fugashi/jieba/pythainlp 형태소 분석기 사전은 모두 이번
세션에 존재하지 않습니다. 이번 세션에 실제로 존재하는 것은 그보다 이전
시점인 **v7 라운드22 스냅샷**(5,612건, `data/v6_r22_snapshot/`)뿐입니다.

그래서 이 노트북은 보고서의 표 2, 그림 3을 재현하려 하지 않습니다. 대신
보고서 7절에 **완전히 설명되어 있는 로직 한 가지**만 독립적으로 재구현합니다:

> "일본어·중국어·태국어 형태소 분석기를 어떤 불릿에 적용할지는 ... 불릿
> '본문' 안에 그 문자 체계(가나·한자·태국 문자)가 실제로 등장하는지로
> 판단한다."

즉 (1) 가나가 있으면 일본어, (2) 가나 없이 한자가 있으면 중국어, (3) 태국
문자가 있으면 태국어, (4) 그 외는 한글/키릴/베트남 성조 확장/일반 라틴
확장/순수 ASCII로 후분류하는 **유니코드 범위 검사 로직**입니다. 이 로직은
비공개 불용어 목록이나 형태소 분석기 없이 순수하게 문자 범위만으로
구현되므로, 보고서에 적힌 설명만으로 정확히 재현할 수 있습니다.

이것을 실제로 복구된 round22 코퍼스(5,612건)에 적용해 문자권별 불릿 수·
비중 표를 만들어 보고서의 표 2와 **나란히(단, 다른 코퍼스 크기·시점임을
명시하며)** 비교합니다. 토큰 총계·어휘 종수·워드클라우드는 실제 불용어
제거·형태소 분석이 필요하므로 이 노트북에서 재현하지 않습니다.


In [1]:

import json
import re
import unicodedata
from pathlib import Path
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")

with open(DATA_DIR / "fandoms_v3_100.json", encoding="utf-8") as f:
    fandoms = json.load(f)

print(f"fandoms_v3_100.json: {len(fandoms)}개 팬덤")

# 보고서와 동일하게 loyalty + spillover 근거문장을 하나의 불릿 리스트로 평탄화
bullets = []
for rec in fandoms:
    for kind in ("loyalty", "spillover"):
        for item in rec.get(kind, []):
            bullets.append({
                "fandom": rec["fandom"],
                "category": rec.get("category"),
                "bullet_type": kind,
                "text": item.get("t", ""),
                "url": item.get("u", ""),
            })

bullets_df = pd.DataFrame(bullets)
print(f"평탄화된 불릿 수: {len(bullets_df)}건  (보고서의 라이브 코퍼스는 10,020건 -- 이 코퍼스는 그보다 이전 시점인 v7 라운드22, 5,612건)")

# 자체 교차검증: v7_progress.json의 current_total과 일치하는지 확인 (기존 노트북들과 동일한 검증 패턴)
with open(DATA_DIR / "v7_progress.json", encoding="utf-8") as f:
    progress = json.load(f)
print(f"v7_progress.json current_total: {progress.get('current_total')}  vs  평탄화 불릿 수: {len(bullets_df)}  일치 =",
      progress.get("current_total") == len(bullets_df))


fandoms_v3_100.json: 100개 팬덤
평탄화된 불릿 수: 5612건  (보고서의 라이브 코퍼스는 10,020건 -- 이 코퍼스는 그보다 이전 시점인 v7 라운드22, 5,612건)
v7_progress.json current_total: 5612  vs  평탄화 불릿 수: 5612  일치 = True



## 1. 문자 범위 기반 스크립트 분기 로직

보고서 1절·7절의 설명을 그대로 코드로 옮깁니다.

- **가나(히라가나·가타카나)** 가 있으면 → 일본어 (`tokenize_ja()`가 실행됐을 것으로 판단)
- 가나는 없고 **한자(CJK 통합 한자)** 가 있으면 → 중국어 (`tokenize_zh()`)
  — 단, 이 우선순위 자체가 보고서 [정정] 사항 그대로입니다: 가나+한자가
  함께 있으면 무조건 일본어로 먼저 판정합니다 (순수 한자 일본어 단어가
  중국어 버킷에 잘못 섞이는 문제를 막기 위함, 보고서 7절 "[정정]" 참조).
- **태국 문자** 가 있으면 → 태국어 (`tokenize_th()`)
- 그 외(=`tokenize_generic()` 만 실행됐을 불릿)는 보고서가 설명한 사후
  재분류 규칙을 그대로 적용합니다: 한글이 있으면 한국어, 키릴 문자가
  있으면 러시아어, 베트남어 성조 결합 문자(라틴 확장 추가 대역)가 있으면
  베트남어, 그 외 악센트가 있는 라틴 확장 문자가 있으면 비영어, 순수 ASCII
  라틴 문자만 있으면 영어.

한 불릿에 여러 문자권이 섞여 있을 수 있으므로(보고서도 "코드스위칭 대응"을
언급합니다), 여기서는 보고서의 "그 함수가 실제로 실행됐는지" 기준을 살려
**우선순위가 있는 단일 버킷 배정**을 그대로 따릅니다(다중 라벨이 아님).


In [2]:

# 유니코드 범위 정의
RE_HANGUL = re.compile(r"[가-힣]")
RE_KANA = re.compile(r"[぀-ゟ゠-ヿ]")          # 히라가나 + 가타카나
RE_HANZI = re.compile(r"[一-鿿]")                       # CJK 통합 한자
RE_THAI = re.compile(r"[฀-๿]")
RE_CYRILLIC = re.compile(r"[Ѐ-ӿ]")                    # 키릴 문자 (Ѐ-ӿ 포함)
# 베트남어 성조 결합 문자를 포함하는 라틴 확장 추가 대역
RE_VIET_EXT = re.compile(r"[Ạ-ỹ]")
# 일반 라틴 확장(악센트 부호): 라틴-1 보충 + 라틴 확장-A/B
RE_LATIN_EXT = re.compile(r"[À-ɏ]")
RE_ASCII_LATIN = re.compile(r"[A-Za-z]")


def classify_bullet(text: str) -> str:
    '''보고서 1절/7절의 tokenize() 분기 로직을 문자 범위 검사로 재현.
    우선순위: 가나 > 한자(가나 없을 때) > 태국 문자 > (일반 경로 사후 재분류)
    '''
    if RE_KANA.search(text):
        return "일본어"
    if RE_HANZI.search(text):
        return "중국어"
    if RE_THAI.search(text):
        return "태국어"
    # 여기부터는 tokenize_generic()만 실행됐을 불릿 -- 사후 재분류
    if RE_HANGUL.search(text):
        return "한국어"
    if RE_CYRILLIC.search(text):
        return "러시아어(키릴문자)"
    if RE_VIET_EXT.search(text):
        return "베트남어(라틴 확장 성조 부호)"
    if RE_LATIN_EXT.search(text):
        return "비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등)"
    if RE_ASCII_LATIN.search(text):
        return "영어"
    return "기타/미분류"


bullets_df["script_bucket"] = bullets_df["text"].apply(classify_bullet)
bullets_df["script_bucket"].value_counts()


script_bucket
한국어                             4470
영어                               833
중국어                              136
일본어                               64
비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등)      39
태국어                               30
베트남어(라틴 확장 성조 부호)                 21
러시아어(키릴문자)                        19
Name: count, dtype: int64


## 2. 문자권별 불릿 수·비중 — round22 코퍼스(5,612건) vs 보고서 표 2(10,020건)

주의: 이 노트북의 분류는 "그 불릿에 어떤 문자권이 **하나라도** 등장했는지"를
보고 있으므로, 보고서 표 2의 "불릿 수(1건 이상)"과 개념은 같지만, 여기서는
우선순위에 따라 **한 불릿을 정확히 하나의 버킷에만** 배정합니다(보고서
표 2는 한 불릿이 여러 언어 버킷에 동시에 잡힐 수 있어 비중 합이 100%를
넘습니다 — 예: 89.2% + 64.3% + ... > 100%). 따라서 절대 수치를 1:1로
대응시키지 말고, **어느 문자권이 코퍼스에서 많이/적게 나타나는지의 방향성**만
비교해야 합니다.


In [3]:

report_table2 = pd.DataFrame([
    {"문자권": "한국어", "report_bullets_10020": 8942, "report_share": 0.892},
    {"문자권": "영어", "report_bullets_10020": 6438, "report_share": 0.643},
    {"문자권": "비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등)", "report_bullets_10020": 158, "report_share": 0.016},
    {"문자권": "러시아어(키릴문자)", "report_bullets_10020": 45, "report_share": 0.004},
    {"문자권": "베트남어(라틴 확장 성조 부호)", "report_bullets_10020": 30, "report_share": 0.003},
    {"문자권": "일본어", "report_bullets_10020": 155, "report_share": 0.016},
    {"문자권": "중국어", "report_bullets_10020": 398, "report_share": 0.040},
    {"문자권": "태국어", "report_bullets_10020": 56, "report_share": 0.006},
])

round22_counts = (
    bullets_df["script_bucket"]
    .value_counts()
    .rename_axis("문자권")
    .reset_index(name="round22_bullets_5612")
)
round22_counts["round22_share"] = (round22_counts["round22_bullets_5612"] / len(bullets_df)).round(4)

comparison = report_table2.merge(round22_counts, on="문자권", how="outer").fillna(0)
comparison = comparison.sort_values("report_bullets_10020", ascending=False).reset_index(drop=True)
print("※ 이 노트북(round22, 단일 버킷 배정)과 보고서(라이브 10,020건, 중복 허용 다중 라벨)는")
print("   집계 방식이 달라 절대 수치가 아니라 방향성만 비교 가능합니다.\n")
comparison


※ 이 노트북(round22, 단일 버킷 배정)과 보고서(라이브 10,020건, 중복 허용 다중 라벨)는
   집계 방식이 달라 절대 수치가 아니라 방향성만 비교 가능합니다.



,문자권,report_bullets_10020,report_share,round22_bullets_5612,round22_share
0,한국어,8942,0.892,4470,0.7965
1,영어,6438,0.643,833,0.1484
2,중국어,398,0.040,136,0.0242
3,비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등),158,0.016,39,0.0069
4,일본어,155,0.016,64,0.0114
5,태국어,56,0.006,30,0.0053
6,러시아어(키릴문자),45,0.004,19,0.0034
7,베트남어(라틴 확장 성조 부호),30,0.003,21,0.0037



## 3. 방향성 관찰

round22 코퍼스(5,612건)에서도 한국어·영어가 압도적 다수이고, 중국어 >
일본어 ≈ 태국어 > 러시아어 > 베트남어 순으로 소수 언어권이 나타나는 큰
흐름은 보고서의 라이브 코퍼스(10,020건)와 방향이 일치하는지 아래 셀에서
직접 확인합니다. 완전히 다른 코퍼스 시점·크기이므로 순위가 정확히 같을
필요는 없습니다 — 여기서는 코퍼스가 계속 성장해도 문자권 구성의 큰 그림이
안정적인지를 참고 삼아 보는 것입니다.


In [4]:

minor_langs = ["일본어", "중국어", "태국어", "러시아어(키릴문자)", "베트남어(라틴 확장 성조 부호)"]
minor_cmp = comparison[comparison["문자권"].isin(minor_langs)].copy()
minor_cmp["report_rank"] = minor_cmp["report_bullets_10020"].rank(ascending=False).astype(int)
minor_cmp["round22_rank"] = minor_cmp["round22_bullets_5612"].rank(ascending=False).astype(int)
minor_cmp["rank_matches"] = minor_cmp["report_rank"] == minor_cmp["round22_rank"]

print(f"소수 언어권 5개 중 순위가 동일한 항목: {minor_cmp['rank_matches'].sum()} / {len(minor_cmp)}")
minor_cmp[["문자권", "report_bullets_10020", "report_rank", "round22_bullets_5612", "round22_rank", "rank_matches"]]


소수 언어권 5개 중 순위가 동일한 항목: 3 / 5


,문자권,report_bullets_10020,report_rank,round22_bullets_5612,round22_rank,rank_matches
2,중국어,398,1,136,1,True
4,일본어,155,2,64,2,True
5,태국어,56,3,30,3,True
6,러시아어(키릴문자),45,4,19,5,False
7,베트남어(라틴 확장 성조 부호),30,5,21,4,False



## 4. 실제 불릿 예시 — 문자권별 1건씩

각 버킷에 실제로 어떤 문장이 배정됐는지 원문 그대로 확인합니다(보고서의
"실제 그 문자가 있는지"로 판정한다는 설명이 실제 데이터에서 타당한지 눈으로
검증하기 위함입니다).


In [5]:

for bucket in ["한국어", "영어", "일본어", "중국어", "태국어", "러시아어(키릴문자)",
               "베트남어(라틴 확장 성조 부호)", "비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등)"]:
    sample = bullets_df[bullets_df["script_bucket"] == bucket]
    if len(sample) == 0:
        print(f"[{bucket}] 해당 버킷 불릿 없음\n")
        continue
    row = sample.iloc[0]
    text_preview = row["text"][:80] + ("..." if len(row["text"]) > 80 else "")
    print(f"[{bucket}] ({row['fandom']}) {text_preview}\n")


[한국어] (이영지) 2026년 서울 올림픽공원 올림픽홀에서 두 번째 월드투어 콘서트 <2.0>을 개최하며 투어 규모를 이어가고 있다.

[영어] (박재범) His 2018 solo concert "ALL OF ME" (Jan 20-21, 2018), his first solo concert in 4...

[일본어] (지드래곤 (G-Dragon)) 후쿠오카 야후옥돔 공연에서 5만 명의 관객이 BIGBANG 컬러 응원봉(노란색·빨간색)으로 도쿄돔 전체를 가득 메웠고, 깜짝 생일 케이크 이벤트...

[중국어] (이영지) 이영지 새 월드투어 첫 개최지를 타이베이로 정했다는 소식을 風傳媒(Storm Media)가 보도.

[태국어] (지드래곤 (G-Dragon)) ชาว V.I.P เตรียมตัวให้พร้อม มาเจอ G-Dragon แลนดิ้งสู่เมืองไทยในงาน"k-star spark ...

[러시아어(키릴문자)] (싸이) Мега-хит стал первым видео в истории, которое превзошло 2 миллиарда просмотров

[베트남어(라틴 확장 성조 부호)] (지드래곤 (G-Dragon)) Ca sĩ khiến khán giả phát cuồng khi hỏi 'Việt Nam, do you love me?'

[비영어(스페인어·프랑스어·포르투갈어·튀르키예어 등)] (지드래곤 (G-Dragon)) Le rappeur coréen est le nouvel ambassadeur du sac "Gabrielle" de Chanel.




## 5. 한계 — 이 노트북이 하지 않는 것

1. **토큰화·불용어 제거·형태소 분석을 하지 않습니다.** 보고서 표 2의
   "토큰 총계"·"어휘 종수"는 STOPWORDS/PARTICLES 제거, fugashi/jieba/
   pythainlp 형태소 분석까지 마친 결과입니다. 이 노트북은 문장 단위
   "어떤 문자권이 있는지"만 봅니다.
2. **자기인용 슬러그 제거(`self_citation_slugs`, v7 77라운드)를 재현하지
   않습니다.** 이 로직은 각 불릿의 출처 URL 도메인에서 슬러그를 추출하는
   구현 세부사항이 보고서에 요약만 되어 있고 전체 코드가 없습니다.
3. **워드클라우드 이미지를 생성하지 않습니다.** 실제 단어 빈도(토큰화 이후)가
   있어야 의미 있는 워드클라우드가 되므로, 위 한계 1번이 해소되기 전까지는
   생성하지 않는 것이 정직하다고 판단했습니다. 보고서의 실제 워드클라우드
   3장은 `docs/TOKENIZER_WORDCLOUD_REPORT.md`에 원본 그대로 첨부되어
   있습니다.
4. **프로즌 스냅샷(v7-40, K=10·M=5·실루엣=0.267) 재현 문제를 다루지
   않습니다.** 보고서 6절이 이미 "0/10 토픽 불일치"를 발견하고 반허구화
   원칙에 따라 재적합을 포기했다고 밝히고 있으며, 이번 세션은애초에 그
   스냅샷 자체를 갖고 있지 않습니다(이번 세션이 가진 것은 v7 라운드22,
   K=8·M=6·실루엣=0.154).
